# GPU setup: CUDA, models, paths (CICT-2026 revision)

Run this in a **Jupyter job** on the SVKM cluster, top to bottom. It is safe to re-run:
anything already in place (Ollama, models, Python packages) is detected and skipped.

**Start the job like this** in Altair Access → Applications → Jupyter:

| Field | Value |
|---|---|
| Container Image | `pytorch_pbs:25.12-py3` |
| Number of Nodes | 1 |
| Number of Processors per Node | **8** (not 1) |
| Number of GPUs per Node | 1 |
| Amount of Memory (MB) | **32000** (not 10) |
| Queue | `workq` |
| Walltime | **long enough for all 14 runs** (they are killed when this job ends) |

What this notebook does (nothing else):

1. Shows the GPU and driver this job was given.
2. Installs Ollama (skipped if already in `/data`).
3. **Connects Ollama to the GPU through CUDA**, proves it from Ollama's own log, and saves
   the paths and GPU settings to `/data/<you>/gpu_env.sh`.
4. Pulls **only the models that are missing**.
5. Runs one real generation and checks `ollama ps` reports `100% GPU`.

The runs themselves are started from a terminal with one `nohup` command.

In [ ]:
# ── Configuration (edit only if your paths differ) ─────────────────────────────
import os, re, glob, json, time, shutil, socket, pathlib, subprocess

os.environ['TMPDIR'] = '/tmp'   # PBS sets a job-specific TMPDIR that does not exist in the container

USER = os.environ.get('USER') or subprocess.run(['id', '-un'], capture_output=True, text=True).stdout.strip() or 'mpstme-dishank'
DATA       = f'/data/{USER}'
PROJ       = f'{DATA}/repo-intel-platform'      # the uploaded project folder
DIST       = f'{DATA}/ollama-dist'              # Ollama install (binary + bundled CUDA libraries)
MODELS_DIR = f'{DATA}/ollama'                   # model weights, persistent between jobs
PACK       = f'{DATA}/context_pack_v2.json'     # the revision context pack
OUT_DIR    = f'{DATA}/evaluation_results'
LOGS       = f'{DATA}/logs'
GPU_ENV    = f'{DATA}/gpu_env.sh'               # written in §3, read by every run

GENERATORS = ['codellama:13b-instruct', 'qwen2.5-coder:14b', 'qwen2.5-coder:32b']
JUDGES     = ['gemma2:9b', 'gemma2:27b', 'mistral:7b-instruct']   # primary + two secondary judges
REQUIRED   = GENERATORS + JUDGES

def sh(cmd, timeout=120):
    # Run a shell command and return stdout+stderr as text (never raises).
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return (r.stdout + r.stderr).strip()
    except Exception as e:
        return f'(failed: {e})'

for d in (DATA, MODELS_DIR, OUT_DIR, LOGS):
    os.makedirs(d, exist_ok=True)
print('user   :', USER)
print('data   :', DATA)
print('project:', PROJ)

## 1. What did this job get?

In [ ]:
print('=== GPU ===')
print(sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader') or '(no nvidia-smi: this job has no GPU)')
print('MIG mode:', sh('nvidia-smi --query-gpu=mig.mode.current --format=csv,noheader') or '(unknown)')
print(sh('nvidia-smi -L'))
print('\nmax CUDA version the driver supports:', (re.findall(r'CUDA Version:\s*([\d.]+)', sh('nvidia-smi')) or ['?'])[0])

print('\n=== GPU visibility inside the container ===')
for k in ['CUDA_VISIBLE_DEVICES', 'NVIDIA_VISIBLE_DEVICES']:
    print(f'  {k} = {os.environ.get(k, "(not set)")}')
print(sh('ls /dev/nvidia* 2>/dev/null') or '  no /dev/nvidia* devices visible')

print('\n=== disk ===')
for p in (DATA, '/tmp'):
    t, u, f = shutil.disk_usage(p)
    print(f'  {p:<24} free {f/2**30:6.1f} GB of {t/2**30:6.1f} GB')

print('\n=== network (only matters if a model is missing) ===')
for host, label in [('registry.ollama.ai', 'Ollama model registry'), ('github.com', 'GitHub (Ollama download)'), ('pypi.org', 'PyPI')]:
    try:
        socket.create_connection((host, 443), timeout=6).close(); print(f'  REACHABLE  {label}')
    except Exception as e:
        print(f'  BLOCKED    {label}  ({type(e).__name__})')

## 2. Install Ollama (skipped if already installed)

The CUDA build of Ollama ships its own CUDA runtime (`lib/ollama/cuda_v12`, ...). It
only needs the host's NVIDIA **driver** library (`libcuda.so.1`), which §3 locates.

In [ ]:
OLLAMA_VERSION = 'v0.33.2'
OLLAMA_ASSET   = 'ollama-linux-amd64.tar.zst'
BIN = f'{DIST}/bin/ollama'

if os.path.exists(BIN):
    print('Ollama already installed:', sh(f'{BIN} --version'))
else:
    os.makedirs(DIST, exist_ok=True)
    uploaded = pathlib.Path(f'{DATA}/{OLLAMA_ASSET}')
    if not uploaded.exists():
        url = f'https://github.com/ollama/ollama/releases/download/{OLLAMA_VERSION}/{OLLAMA_ASSET}'
        print('Downloading', url)
        print(sh(f'curl -fL {url} -o {uploaded}', timeout=1800))
    print(sh(f'tar --zstd -xf {uploaded} -C {DIST}/', timeout=900))
    print('Installed:', sh(f'{BIN} --version'))

print('\nBundled GPU runtimes:', sorted(os.path.basename(p) for p in glob.glob(f'{DIST}/lib/ollama/*') if os.path.isdir(p)))

## 3. Connect Ollama to the GPU (CUDA)

The three things that make Ollama silently fall back to the CPU, and what this cell does
about each:

1. **The driver library isn't on the library path.** Containers mount `libcuda.so.1` in
   different places; this cell finds it and adds its folder to `LD_LIBRARY_PATH`.
2. **Ollama's own CUDA runtime isn't on the path.** `ollama-dist/lib/ollama` is added too.
3. **A MIG slice isn't selected.** On MIG, the slice is pinned with its `MIG-…` UUID.

The result is saved to `gpu_env.sh`, which the run command loads first, so the battery
and rejudge scripts start Ollama with exactly these settings.

The proof is Ollama's own startup log: its `inference compute` line must say
`library=cuda` (or CUDA), not `cpu`.

In [ ]:
# 1. Find the NVIDIA driver library inside this container.
candidates = []
for line in sh('ldconfig -p 2>/dev/null | grep libcuda.so.1').splitlines():
    m = re.search(r'=>\s*(\S+)', line)
    if m: candidates.append(m.group(1))
for d in ['/usr/lib/x86_64-linux-gnu', '/usr/lib64', '/usr/local/nvidia/lib64', '/usr/local/nvidia/lib',
          '/usr/local/cuda/compat', '/usr/local/cuda/lib64', '/usr/lib/wsl/lib']:
    candidates += glob.glob(f'{d}/libcuda.so.1')
driver_dirs = []
for c in candidates:
    d = os.path.dirname(os.path.realpath(c))
    if d not in driver_dirs: driver_dirs.append(d)
print('libcuda.so.1 found in:', driver_dirs or 'NOWHERE -- the GPU is not mounted into this container (check "Number of GPUs per Node")')

# 2. Ollama's bundled CUDA runtime.
ollama_libs = f'{DIST}/lib/ollama'
print('Ollama CUDA runtimes  :', sorted(os.path.basename(p) for p in glob.glob(f'{ollama_libs}/cuda*')) or 'NONE (wrong Ollama build?)')

# 3. MIG: pin the slice this job was given.
cvd = os.environ.get('CUDA_VISIBLE_DEVICES', '')
mig_uuids = re.findall(r'\(UUID:\s*(MIG-[^)]+)\)', sh('nvidia-smi -L'))
if mig_uuids and not cvd.startswith('MIG-'):
    cvd = mig_uuids[0]
    print('MIG slice pinned      :', cvd)
elif cvd:
    print('CUDA_VISIBLE_DEVICES  :', cvd)

ld_path = ':'.join([ollama_libs] + driver_dirs + [p for p in os.environ.get('LD_LIBRARY_PATH', '').split(':') if p])
env = {
    'PATH': f'{DIST}/bin:/usr/local/bin:' + os.environ.get('PATH', ''),
    'LD_LIBRARY_PATH': ld_path,
    'OLLAMA_MODELS': MODELS_DIR,
    'OLLAMA_HOST': 'http://127.0.0.1:11434',
    'OLLAMA_NUM_CTX': '8192',            # pinned: changing it breaks comparability with the study
    'OLLAMA_MAX_LOADED_MODELS': '2',     # generator + judge resident together
    'OLLAMA_NUM_PARALLEL': '1',          # deterministic: no request batching
    'OLLAMA_KEEP_ALIVE': '30m',
}
if cvd:
    env['CUDA_VISIBLE_DEVICES'] = cvd
os.environ.update(env)

with open(GPU_ENV, 'w') as f:
    f.write('# Written by 00_setup_gpu_image.ipynb section 3. Loaded by the nohup run command.\n')
    for k, v in env.items():
        f.write(f'export {k}="{v}"\n')
print('\nSaved GPU settings to', GPU_ENV)

def start_ollama(log='/tmp/ollama_serve.log'):
    # (Re)start the Ollama server with the GPU settings above; return the startup log.
    sh("pkill -f 'ollama serve'"); time.sleep(2)
    subprocess.Popen(f'nohup ollama serve > {log} 2>&1 &', shell=True, env=os.environ.copy())
    for _ in range(60):
        if 'NAME' in sh('ollama list', timeout=10): break
        time.sleep(1)
    time.sleep(2)
    return open(log).read()

log = start_ollama()
gpu_lines = [l for l in log.splitlines() if re.search(r'inference compute|no compatible gpu|library=|cuda driver|cudaGetDevice|failed to load', l, re.I)]
print('\n=== Ollama startup: GPU discovery ===')
print('\n'.join(gpu_lines[-12:]) or '(no GPU discovery lines; full log below)\n' + log[-3000:])

found_gpu = any(re.search(r'library=(cuda|CUDA)', l) for l in gpu_lines)
no_gpu = any(re.search(r'no compatible gpu|library=cpu', l, re.I) for l in gpu_lines)
print('\nRESULT:', 'GPU DETECTED by Ollama (CUDA)' if found_gpu and not no_gpu else
      'NO GPU -- Ollama is on the CPU. Fix this before running anything (see the checklist below).')

**If the result is NO GPU**, check in this order:

- §1 shows no `/dev/nvidia*` or `nvidia-smi` fails → the job has no GPU. Resubmit with
  *Number of GPUs per Node = 1*.
- `libcuda.so.1 found in: NOWHERE` → same cause: the driver isn't mounted.
- `Ollama CUDA runtimes: NONE` → the wrong tarball was installed. Delete `ollama-dist`
  and re-run §2 (`ollama-linux-amd64.tar.zst` is the CUDA build).
- The log mentions a CUDA/driver version mismatch → the driver (§1, "max CUDA version")
  is older than Ollama's runtime. Set `OLLAMA_VERSION` in §2 to an older release
  (for example `v0.6.8`), delete `ollama-dist`, and re-run §2 and §3.

## 4. Models: pull only what's missing

In [ ]:
installed = {l.split()[0] for l in sh('ollama list').splitlines()[1:] if l.strip()}
missing = [m for m in REQUIRED if m not in installed]
for m in REQUIRED:
    print(f'  {"OK     " if m in installed else "MISSING"}  {m}')

if not missing:
    print('\nAll required models are already in', MODELS_DIR)
else:
    try:
        socket.create_connection(('registry.ollama.ai', 443), timeout=6).close()
        for m in missing:
            print(f'\n--- pulling {m} ---')
            !ollama pull {m}
    except Exception:
        print('\nThe Ollama registry is BLOCKED from this node. Pull these on a machine with internet')
        print(f'and upload its models folder into {MODELS_DIR}:', ', '.join(missing))
    still = [m for m in REQUIRED if m not in {l.split()[0] for l in sh('ollama list').splitlines()[1:] if l.strip()}]
    print('\nStill missing:', still or 'none')

## 5. Smoke test: the heaviest real load, fully on the GPU

The `qwen2.5-coder:32b` runs keep the 32B generator **and** the `gemma2:9b` judge loaded
at the same time (~26 GB). If that doesn't fit, Ollama silently splits the model between
GPU and CPU and those runs take many times longer. This loads both together, exactly as
a run does, and requires **both at 100% GPU** in `ollama ps`. Takes a few minutes, most
of it loading the 32B model.

In [ ]:
def smoke(model):
    # One short generation; returns (reply, eval rate, load time, wall clock seconds).
    t0 = time.time()
    out = sh(f"ollama run {model} 'Reply with exactly: OK' --verbose", timeout=1800)
    wall = time.time() - t0
    pick = lambda pat: next((l.split(':', 1)[1].strip() for l in out.splitlines() if re.match(pat, l.strip())), '?')
    reply = next((l.strip() for l in out.splitlines() if l.strip() and not re.match(r'^(total|load|prompt|eval)', l.strip())), out[-300:])
    return reply, pick(r'^eval rate'), pick(r'^load duration'), wall

results = {}
for m in ['gemma2:9b', 'qwen2.5-coder:32b']:     # the judge, then the largest generator
    print(f'--- {m} ---')
    reply, rate, load, wall = smoke(m)
    results[m] = rate
    print(f'  reply        : {reply}')
    print(f'  eval rate    : {rate}')
    print(f'  load duration: {load}')
    print(f'  wall clock   : {wall:.1f}s')

ps = sh('ollama ps')
print('\n' + ps)
print('\nGPU memory / utilisation:', sh('nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv,noheader'))

rows = {l.split()[0]: l for l in ps.splitlines()[1:] if l.strip()}
problems = []
for m in ['gemma2:9b', 'qwen2.5-coder:32b']:
    if m not in rows:
        problems.append(f'{m} is not loaded (it was evicted: both must stay resident together)')
    elif '100% GPU' not in rows[m]:
        problems.append(f'{m} is not fully on the GPU: {" ".join(rows[m].split()[3:6])}')

if not problems:
    print('\nRESULT: PASS -- the 32B generator and the judge are both fully on the GPU, together.')
else:
    print('\nRESULT: FAIL')
    for p in problems: print('  -', p)
    print('  If this is a CPU/GPU split, the GPU memory given to this job is too small for')
    print('  32B + judge. If 0% GPU, go back to section 3. Do not start the runs until this passes.')

## Done: start the runs from a terminal

Setup is complete when **§3 says GPU DETECTED** and **§5 says PASS**. The GPU settings
are saved in `/data/<you>/gpu_env.sh`, which `run_all.sh` loads before every job.

Open **File → New → Terminal** and run (one line):

```bash
cd /data/$USER/repo-intel-platform && JUDGES="mistral:7b-instruct gemma2:27b" nohup setsid bash ./hpc/run_all.sh revision > /data/$USER/run_all.boot.log 2>&1 & disown
```

It runs the 3 main batteries, then both judges on them, then the 9 variance batteries,
one after another. It keeps running after you close the terminal or the browser, but
**not after this Jupyter job ends**, so start the job with a long enough walltime. If
the walltime does run out, start a new job and run the same line: finished runs are
skipped.

Watch it with: `tail -f /data/$USER/evaluation_results/run_all.log`